# ViLT：Vision-and-Language Transformer（VQA）

这个 Notebook 展示 `ViLT` 在 Visual Question Answering（视觉问答）任务上的完整推理与分析流程。

内容包括：
- ViLT 架构解读：图像与文本 token 拼接进单一 Transformer
- 与 CLIP（双塔检索）的核心区别
- Processor 编码与形状分析
- 单图多问推理演示
- Attention 权重可视化
- ViLT 的适用场景与局限

## 1. 环境准备

```bash
pip install torch transformers pillow requests matplotlib
```

In [ ]:
from dataclasses import dataclass
from io import BytesIO

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import requests
import torch
from PIL import Image
from transformers import ViltForQuestionAnswering, ViltProcessor

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    model_name: str = 'dandelin/vilt-b32-finetuned-vqa'
    # 返回注意力权重，用于可视化文本关注哪些图像区域
    output_attentions: bool = True

cfg = Config()
cfg

## 2. 加载示例图像

In [ ]:
def load_image_from_url(url):
    response = requests.get(url, timeout=10)
    return Image.open(BytesIO(response.content)).convert('RGB')


# 使用公开图片做演示
image_urls = [
    'http://images.cocodataset.org/val2017/000000039769.jpg',   # 猫咪
    'http://images.cocodataset.org/val2017/000000283900.jpg',   # 厨房
]

images = [load_image_from_url(url) for url in image_urls]

fig, axes = plt.subplots(1, len(images), figsize=(12, 5))
for ax, img in zip(axes, images):
    ax.imshow(img)
    ax.axis('off')
plt.suptitle('演示图像')
plt.tight_layout()
plt.show()

## 3. 模型与 Processor 加载

In [ ]:
processor = ViltProcessor.from_pretrained(cfg.model_name)
model = ViltForQuestionAnswering.from_pretrained(cfg.model_name).to(device)
model.eval()

print(f'模型加载完成，VQA 答案词表大小：{model.config.num_labels}')

## 4. ViLT 结构解读

### 4.1 与 CLIP 的本质区别

| 维度 | CLIP | ViLT |
|------|------|------|
| 架构 | 双塔（图像塔 + 文本塔） | 单塔（图文共享同一 Transformer） |
| 融合时机 | 晚期（两个独立编码器输出做点积） | 早期（图文 token 拼接后一起过 Transformer） |
| 适合任务 | 图文检索、zero-shot 分类 | VQA、图文推理 |
| 视觉 Backbone | 需要强视觉编码器（ViT-L/14） | **无独立视觉 Backbone**，直接用 patch embedding |
| 优势 | 可扩展到亿级数据，zero-shot 能力强 | 更深度的跨模态融合，推理能力更强 |

### 4.2 Token 拼接方式

```
[CLS] text_token1 text_token2 ... [SEP] img_patch1 img_patch2 ... img_patchN [SEP]
```

图像 patch 和文本 token **在同一序列中共同参与 Self-Attention**，使文本能直接关注到对应的图像区域。

### 4.3 位置编码
- 文本：普通 1D 位置编码
- 图像 patch：2D 位置编码（行 + 列），保留空间信息

## 5. Token 形状分析

In [ ]:
@torch.no_grad()
def inspect_shapes(model, processor, image, question, device):
    inputs = processor(image, question, return_tensors='pt').to(device)

    print('=== Processor 输出形状 ===')
    for k, v in inputs.items():
        print(f'  {k:25s} -> {tuple(v.shape)}')

    outputs = model(**inputs, output_attentions=True)
    print(f'\nlogits                    -> {tuple(outputs.logits.shape)}  (batch, num_answer_classes)')
    print(f'attentions[0]（第1层）    -> {tuple(outputs.attentions[0].shape)}  (batch, heads, seq, seq)')


inspect_shapes(model, processor, images[0], 'What animal is in the image?', device)

## 6. 单图多问推理演示

In [ ]:
@torch.no_grad()
def vqa(model, processor, image, questions, device, topk=3):
    results = []
    for q in questions:
        inputs = processor(image, q, return_tensors='pt').to(device)
        outputs = model(**inputs)
        probs = outputs.logits.softmax(dim=-1)[0]
        top_ids = probs.argsort(descending=True)[:topk]
        top_answers = [(model.config.id2label[i.item()], probs[i].item()) for i in top_ids]
        results.append((q, top_answers))
    return results


# 对第一张图（猫）提问
questions_cat = [
    'What animal is in the image?',
    'How many cats are there?',
    'What color is the cat?',
    'Is the cat sleeping?',
]

results = vqa(model, processor, images[0], questions_cat, device)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(images[0])
ax.axis('off')
ax.set_title('VQA 演示图像')
plt.show()

for question, answers in results:
    print(f'Q: {question}')
    for ans, prob in answers:
        print(f'   {ans:20s} {prob:.3f}')
    print()

## 7. Attention 可视化

取最后一层的 CLS token 对图像 patch 的注意力权重，可以粗略看出模型在回答问题时关注了哪些区域。

In [ ]:
@torch.no_grad()
def visualize_attention(model, processor, image, question, device):
    inputs = processor(image, question, return_tensors='pt').to(device)
    outputs = model(**inputs, output_attentions=True)

    # 取最后一层、所有头的平均注意力
    last_attn = outputs.attentions[-1][0].mean(dim=0)  # seq x seq

    num_text_tokens = (inputs['input_ids'] != processor.tokenizer.pad_token_id).sum().item()
    # CLS token (index 0) 对图像 patch 的注意力
    img_attn = last_attn[0, num_text_tokens:].cpu().numpy()

    # 尝试重组成 2D 热图
    n = len(img_attn)
    h = w = int(n ** 0.5)
    if h * w != n:
        print(f'patch 数量 {n} 非完全平方，跳过热图展示')
        return

    attn_map = img_attn.reshape(h, w)
    img_resized = image.resize((224, 224))

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_resized)
    axes[0].set_title(f'Q: {question}')
    axes[0].axis('off')

    im = axes[1].imshow(attn_map, cmap='hot', interpolation='bilinear')
    axes[1].set_title('CLS → patch 注意力权重')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046)

    plt.tight_layout()
    plt.show()


visualize_attention(model, processor, images[0], 'What animal is in the image?', device)

## 8. ViLT 的适用场景与局限

### 优势
- 无需独立视觉 Backbone（如 Faster-RCNN 提取区域特征），推理速度更快
- 图文早期融合，对需要细粒度跨模态理解的任务（VQA、图文推理）效果好

### 局限
- 由于没有专门的视觉编码器，纯视觉表示能力不如 CLIP（ViT-L/14 backbone）
- 对图像分辨率敏感，高分辨率输入会导致 token 序列过长，计算量激增
- VQA 任务答案空间固定（closed-set），无法开放式生成答案（需要 BLIP 等生成模型）

In [ ]:
# 对第二张图（厨房）提问
questions_kitchen = [
    'What room is this?',
    'What appliance is visible?',
    'Is there food on the counter?',
]

results2 = vqa(model, processor, images[1], questions_kitchen, device)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(images[1])
ax.axis('off')
ax.set_title('VQA 演示图像 2')
plt.show()

for question, answers in results2:
    print(f'Q: {question}')
    for ans, prob in answers:
        print(f'   {ans:20s} {prob:.3f}')
    print()